# PDF integrator for HDAB id project

This script generates PDF reports to make it easier to decide on integrative identifications that combine morphological and DNA data.

Dependencies (all are available through conda):
- PyPDF v5.0 or higher (lower versions don't support functions to reduce PDF file size)
- PsycoPG
- Pandas
- ReportLab
- Pillow
- SQLalchemy

## Import dependencies, fonts, create output directories

In [1]:
import os
import io
import time
import math
import sys
import linecache
import psycopg2
import pandas as pd
import reportlab
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.utils import ImageReader
from pypdf import PdfWriter
from PIL import Image
from datetime import date
from pathlib import Path
from sqlalchemy import create_engine

# Add Arial as a font for the PDF
font_dir = "./fonts/" 
arial_reg = os.path.join(font_dir, "Arial.ttf")
arial_bold = os.path.join(font_dir, "Arial Bold.ttf")
arial_italic = os.path.join(font_dir, "Arial Italic.ttf")
arial_bolditalic = os.path.join(font_dir, "Arial Bold Italic.ttf")
pdfmetrics.registerFont(TTFont('Arial', arial_reg))
pdfmetrics.registerFont(TTFont('Arial-Bold', arial_bold))
pdfmetrics.registerFont(TTFont('Arial-Italic', arial_italic))
pdfmetrics.registerFont(TTFont('Arial-BoldItalic', arial_bolditalic))

# Create output directories if they do not yet exist
individualreports_dir = "Individual_reports"
os.makedirs('Individual_reports', exist_ok=True)
os.makedirs('Combined_reports', exist_ok=True)

# Set script parameters

In [2]:
# specify the postgresql database name to pull information from
database = 'aiven_hdab'

# specify directory with the specimen images
img_dir = "/Volumes/Rubinoff_nas_412/DNA_voucher_photos/HDOA_Vouchers/"
#img_dir = "./test_imgs/"

# keep individual reports? True/False
keep_ind_reports = False

# target resolution for specimen images [horizontal width in pixels]
specimen_img_resolution = 800 

## PostgreSQL connection test function 

In [14]:
def psqlconnectiontest(database):
    connectstringfile = f"./.connectstring_{database}"
    if not os.path.exists(connectstringfile):
        sys.exit(f"Missing .connectstring file for database {database} : stopping")
    connectstring = linecache.getline(connectstringfile, 1).rstrip('\n')
    try:
        engine = create_engine(connectstring)
        with engine.connect() as conn:
            print(f"Successfully connected to psql database: {database}")
        return engine # return engine so you can pass it directly to pd.read_sql_query()
    
    except Exception as e:
        sys.exit(f"Could not connect to psql database: stopping\nError: {e}")

## Function to create a ReportLab PDF with info from dataframe and add images

In [21]:
def pdfintegrator(df, individualreports_dir, img_dir, specimen_img_resolution):
    for row in df.sort_values(by=['hdoareference', 'morphospeciescode']).itertuples():
        # Create canvas with morphospeciescode filename in subfolder Reports
        morphospeciescode = row.morphospeciescode
        print("Creating PDF report for: " + (morphospeciescode))
        filename = str(morphospeciescode) + ".pdf"
        full_path = os.path.join(individualreports_dir, filename)
        c = canvas.Canvas(full_path, pagesize=letter) # Letter size; width: 612, height: 792 points
        page_width, page_height = letter
        
        # Add morphospecies code as title
        x_center = page_width / 2 # to center text on page
        c.setFont("Arial-Bold", 14) # font and size
        c.drawCentredString(x_center, 720, f"Morphospecies: {row.morphospeciescode}")
        
        # Add information from dataframe
        c.setFont("Arial", 11) # keep 12 y value spacing between text with font size 11
        c.drawString(48, 685, f"HDAB reference: {row.hdoareference}")
        c.drawString(306, 685, f"HDAB ID: {row.hdoaprelimid}")
        c.drawString(48, 673, f"Date collected: {row.datecollected}")
        c.drawString(48, 661, f"Host/source verbatim: {row.hostverbatim}")
        c.drawString(48, 649, f"Origin verbatim: {row.originverbatim}")

        c.drawString(48, 625, f"Morph. ID order: {row.morphidorder}")
        c.drawString(306, 625, f"Inf. morph. ID: {row.morphidinformal}")
        c.drawString(48, 613, f"Morph. ID family: {row.morphidfamily}")
        c.drawString(306, 613, f"Morph. ID species: {row.morphidbispecies}")
        c.drawString(48, 601, f"Morph. ID identifier: {row.identifier}")
        c.drawString(48, 589, f"Morph. ID notes: {row.idnotes}")
        
        c.drawString(48, 565, f"Extraction code: {row.extractioncode}")
        c.drawString(306, 565, f"DNA ID: {row.boldmatch}")
        c.drawString(48, 553, f"DNA ID notes: {row.pcrnotes}")
        
        c.drawString(48, 529, f"Int. ID order: {row.finalorder}")
        c.drawString(306, 529, f"Integrative ID: {row.finalid}")
        c.drawString(48, 517, f"Int. ID family: {row.finalfamily}")
        c.drawString(306, 517, f"Common name: {row.finalcommonname}")
        
        # Add photo of morphotype
        # get a list of photos in the img_dir
        jpg_filenames = [
            f for f in os.listdir(img_dir) 
            if f.lower().endswith('.jpg') and os.path.isfile(os.path.join(img_dir, f))
        ]
        # Select image file names containing "morphospeciescode"
        morphospeciesimages = [item for item in jpg_filenames if morphospeciescode in item]
        if len(morphospeciesimages) == 0:
            print("   No images found, continuing without image")
        else:
            speciesimage = os.path.join(img_dir, morphospeciesimages[0]) # selects first photo in the list
            
            # reduce image size on the fly without creating more files
            target_width = specimen_img_resolution
            with Image.open(speciesimage) as img:
                w_percent = target_width / float(img.size[0]) # Calculate aspect ratio
                target_height = int((float(img.size[1]) * float(w_percent)))          
                resized_img = img.resize((target_width, target_height), Image.Resampling.LANCZOS)
                img_buffer = io.BytesIO() # Save the resized image to an in-memory byte buffer
                resized_img.save(img_buffer, format="JPEG", quality=85) # JPEG or PNG etc
                img_buffer.seek(0)  # Reset buffer pointer to the beginning
            
            # Print on page using reportlab (passing the buffer object instead of a file path)
            reportlab_image = ImageReader(img_buffer) # wrap the buffer in ImageReader
            img_width = 500
            img_height = 400
            x_coord = (page_width - img_width) / 2
            c.drawImage(reportlab_image, x_coord, 10, width=img_width, height=img_height, preserveAspectRatio=True)
            
            # Add image filename underneath
            imgfilename = os.path.basename(speciesimage)
            c.setFont("Arial", 10) # font and size
            c.drawCentredString(x_center, 60, imgfilename)
            
        c.showPage() # finish page
        c.save() # construct and save file to .pdf
        print("   Successfully created report " + str(filename))

## Function to merge individual PDFs into one in separate folder

In [22]:
def combinepdf(individualreports_dir):
    today = date.today()
    pdf_with_folder = [
        os.path.join(individualreports_dir, f) 
        for f in os.listdir(individualreports_dir) 
        if f.lower().endswith('.pdf') and os.path.isfile(os.path.join(individualreports_dir, f))
    ]
    writer = PdfWriter()
    ordered_pdf_with_folder = sorted(pdf_with_folder)
    for pdf in ordered_pdf_with_folder:
        writer.append(pdf)

    writer.compress_identical_objects(remove_duplicates=True, remove_unreferenced=True)
    writer.write("./Combined_reports/" + today.strftime("%y%m%d") + "_combined_report.pdf")
    writer.close()
    print(f"Successfully merged {len(pdf_with_folder)} files into one PDF")

## Run functions

In [24]:
start_time = time.perf_counter()

# test connection with database and return SQLAlchemy connection engine if successful
engine = psqlconnectiontest(database)
 
# get data from psql database
sql = "SELECT * FROM public.integratorview;"
df = pd.read_sql_query(sql, engine)
conn = None # close connection

# generate a PDF report for each morphospecies
try:
    pdfintegrator(df, individualreports_dir, img_dir, specimen_img_resolution)
except Exception as e:
    print(f"An error occurred: {e}")

# combine reports into one PDF
try:
    combinepdf(individualreports_dir)
except Exception as e:
    print(f"An error occurred: {e}")

# remove individual reports if requested
if keep_ind_reports == False:
    print("Cleaning up, deleting individual PDFs")
    ind_reports_path = Path("./Individual_reports")
    for file in ind_reports_path.glob("*.pdf"):
        file.unlink(missing_ok=True) # missing_ok=True prevents errors if the file disappears mid-process
    
end_time = time.perf_counter()
execution_time_seconds = end_time - start_time
minutes, seconds = divmod(execution_time_seconds, 60)
print(f"Execution time: {int(minutes)}m {seconds:.2f}s")

Successfully connected to psql database: aiven_hdab
Creating PDF report for: HDOA260619_023_M01
   Successfully created report HDOA260619_023_M01.pdf
Creating PDF report for: HDOA260619_023_M02
   Successfully created report HDOA260619_023_M02.pdf
Creating PDF report for: HDOA260619_027_M01
   Successfully created report HDOA260619_027_M01.pdf
Creating PDF report for: HDOA260619_027_M02
   Successfully created report HDOA260619_027_M02.pdf
Creating PDF report for: HDOA260619_027_M03
   Successfully created report HDOA260619_027_M03.pdf
Creating PDF report for: HDOA260619_027_M04
   Successfully created report HDOA260619_027_M04.pdf
Creating PDF report for: HDOA260619_027_M05
   Successfully created report HDOA260619_027_M05.pdf
Creating PDF report for: HDOA260619_020_M01
   Successfully created report HDOA260619_020_M01.pdf
Creating PDF report for: HDOA260619_020_M02
   Successfully created report HDOA260619_020_M02.pdf
Creating PDF report for: HDOA260619_021_M01
   Successfully create